
# DRISHTI D-DAY — Multi-HMR End-to-End Kaggle Pipeline

This notebook is deliberately built as a **single-run, resumable Kaggle pipeline** for the Multi-HMR / SMPL-X branch.

## What it does

1. Reuses or bootstraps the isolated Python 3.9 Multi-HMR environment.
2. Pins the frozen Multi-HMR repository commit.
3. Installs only inference dependencies required by this branch.
4. Finds the **full SMPL-X v1.1 `SMPLX_NEUTRAL.npz`** from attached private Kaggle inputs.
5. Downloads/loads the **pretrained `multiHMR_672_S`** checkpoint.
6. Runs a real smoke inference.
7. Runs Multi-HMR over the Drishti videos at configurable sampling FPS.
8. Persists per-person 3D/pose features and frame-level aggregates.
9. Uses simple temporal association for stable motion features.
10. Optionally merges external object evidence later (phone/mouse/monitor/paper probabilities).
11. Builds temporal rolling features.
12. Trains downstream suspicious-behavior classifiers **only from real temporal annotations**.
13. Splits by **video**, not random frames, to reduce leakage.
14. Tunes a decision threshold on validation data.
15. Tests frame metrics, event metrics, false positives/minute, inference FPS and VRAM.
16. Saves the trained model, threshold, predictions, events and metrics.

## Important scope decision

**Multi-HMR itself is not trained from scratch here.** It is a pretrained 3D human reconstruction provider.  
The trainable component is the downstream **temporal behavior classifier** over the extracted evidence.

That is intentional for D-Day: training Multi-HMR from scratch on the eight CCTV videos would be the wrong use of time and data.

## One thing the notebook will never fabricate

You still need real event intervals for supervised training, for example:

`video,start_sec,end_sec,label`

If no annotation CSV is attached and `MANUAL_EVENTS` is empty, the notebook will still finish expensive Multi-HMR extraction and produce an annotation template, but supervised training will be skipped instead of inventing labels.


In [ ]:

# ============================================================
# CELL 1 — D-DAY CONFIG
# EDIT ONLY THIS CELL IF YOU NEED TO CHANGE RUN BEHAVIOR
# ============================================================

from pathlib import Path

# Frozen provider
REPO = Path("/kaggle/working/multi-hmr")
REPO_COMMIT = "651fb411e1cbcc626aaa5f38805ecab9cc891f7a"

ENV = Path("/kaggle/working/envs/multihmr_py39")
ENV_PY = ENV / "bin" / "python"

# Drishti paths
OUTPUT_ROOT = Path("/kaggle/working/drishti_multihmr")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

PREFERRED_VIDEO_ROOT = Path("/kaggle/input/datasets/dravvalt/drishti-dataset")

# Provider config
MODEL_NAME = "multiHMR_672_S"   # fastest official 672 model; use B/L only after baseline works
DET_THRESH = 0.30
NMS_KERNEL_SIZE = 3
FOV_DEG = 60.0

# D-Day extraction config
SAMPLE_FPS = 1.0               # increase to 2.0 later if runtime budget allows
DEBUG_EVERY_N_SAMPLES = 600    # lightweight 2D-joint debug frame; 0 disables
RAW_EVERY_N_SAMPLES = 300      # compressed raw j3d/j2d/rotvec snapshot; 0 disables

# Parallel inference: auto-uses all visible GPUs up to this cap.
MAX_GPU_WORKERS = 2

# Temporal classifier
TEMPORAL_WINDOW_SEC = 5.0
PERSISTENCE_SEC = 2.0
MAX_FP_PER_MIN_TARGET = 1.0

# Annotation semantics.
# IMPORTANT: if True, all time outside annotated intervals *within annotated videos*
# is treated as normal. Only keep True if you annotated all suspicious intervals.
ASSUME_OUTSIDE_ANNOTATIONS_NORMAL = True

NORMAL_LABELS = {
    "normal",
    "board_look",
    "invigilator_look",
    "normal_look_around",
    "stretching",
    "writing",
}

# Optional: paste real intervals here instead of uploading a CSV.
# Do NOT invent these.
MANUAL_EVENTS = [
    # {"video": "03.CCTV Mobile Usage.mkv", "start_sec": 10.0, "end_sec": 18.0, "label": "phone_use"},
]

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("MODEL_NAME:", MODEL_NAME)
print("SAMPLE_FPS:", SAMPLE_FPS)


In [ ]:

# ============================================================
# CELL 2 — IDEMPOTENT ENVIRONMENT + FROZEN REPO BOOTSTRAP
# THIS REPLACES THE OLD CELL 23/24/25 CHAIN
# ============================================================

import os
import sys
import shutil
import subprocess
from pathlib import Path

def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), check=True, **kwargs)

# ------------------------------------------------------------
# 1) Frozen repo
# ------------------------------------------------------------
if not REPO.exists():
    run(["git", "clone", "https://github.com/naver/multi-hmr.git", REPO])
    run(["git", "-C", REPO, "checkout", REPO_COMMIT])
else:
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
    ).strip()
    if head != REPO_COMMIT:
        raise RuntimeError(
            f"Existing repo is at {head}, expected {REPO_COMMIT}. "
            "Do not overwrite an unknown working tree during D-Day."
        )
    print("Frozen repo already correct:", head)

# ------------------------------------------------------------
# 2) System GLU check
# ------------------------------------------------------------
glu_present = subprocess.run(
    ["bash", "-lc", "ldconfig -p | grep -q 'libGLU.so.1'"],
).returncode == 0

if not glu_present:
    run(["apt-get", "update"])
    run(["apt-get", "install", "-y", "libglu1-mesa", "libgl1", "libegl1"])
else:
    print("GLU already present — apt install skipped.")

# ------------------------------------------------------------
# 3) Create isolated Python 3.9 env only if missing
# ------------------------------------------------------------
if not ENV_PY.exists():
    if shutil.which("uv") is None:
        run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    uv = shutil.which("uv")
    if uv is None:
        candidate = Path(sys.executable).parent / "uv"
        uv = str(candidate) if candidate.exists() else None
    assert uv is not None, "uv installation failed."
    run([uv, "python", "install", "3.9"])
    ENV.parent.mkdir(parents=True, exist_ok=True)
    run([uv, "venv", str(ENV), "--python", "3.9", "--seed"])

# ------------------------------------------------------------
# 4) Health check. If it already passes, do not touch packages.
# ------------------------------------------------------------
health_code = r'''
import torch, torchvision, numpy, xformers
import roma, PIL, pyrender, einops, smplx, trimesh, pyglet, scipy, cv2
assert torch.__version__.startswith("2.0.1")
assert torchvision.__version__.startswith("0.15.2")
assert numpy.__version__ == "1.22.4"
assert xformers.__version__ == "0.0.20"
assert torch.cuda.is_available()
print("ENV_HEALTH=PASS")
'''

health = subprocess.run(
    [str(ENV_PY), "-c", health_code],
    capture_output=True,
    text=True,
)

if health.returncode == 0:
    print(health.stdout)
    print("Existing inference environment reused. No pip reinstall.")
else:
    print("Environment health check did not pass; bootstrapping pinned inference stack.")
    print(health.stderr[-3000:])

    # Foundation
    run([
        ENV_PY, "-m", "pip", "install",
        "--index-url", "https://download.pytorch.org/whl/cu118",
        "torch==2.0.1", "torchvision==0.15.2", "torchaudio==2.0.2",
    ])
    run([ENV_PY, "-m", "pip", "install", "numpy==1.22.4"])
    run([ENV_PY, "-m", "pip", "install", "xformers==0.0.20", "--no-deps"])

    constraints = ENV / "drishti_multihmr_constraints.txt"
    constraints.write_text(
        "\n".join([
            "torch==2.0.1",
            "torchvision==0.15.2",
            "torchaudio==2.0.2",
            "numpy==1.22.4",
            "xformers==0.0.20",
        ])
    )

    # Lean inference set. pyvista/vtk are deliberately not required.
    run([
        ENV_PY, "-m", "pip", "install", "-c", constraints,
        "roma",
        "pillow==10.0.1",
        "pyrender==0.1.45",
        "einops==0.6.1",
        "smplx",
        "trimesh==3.22.3",
        "pyglet==1.5.24",
        "tqdm==4.65.0",
        "scipy",
        "opencv-python-headless==4.8.1.78",
        "ipdb",
    ])

    # Final health check
    run([ENV_PY, "-c", health_code])

# ------------------------------------------------------------
# 5) Direct frozen-model import
# ------------------------------------------------------------
model_check = r'''
import os, sys, torch
os.chdir("/kaggle/working/multi-hmr")
sys.path.insert(0, "/kaggle/working/multi-hmr")
import model
import blocks.smpl_layer
assert torch.cuda.is_available()
print("MULTIHMR_CORE_IMPORT=PASS")
print("GPU:", torch.cuda.get_device_name(0))
'''

run([ENV_PY, "-c", model_check])

print("\n✅ D-DAY ENVIRONMENT READY")


In [ ]:

# ============================================================
# CELL 3 — AUTO-SELECT FULL SMPL-X ASSET + DISCOVER VIDEOS
# ============================================================

import shutil
import numpy as np
import cv2
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1) Find and validate the full SMPL-X v1.1 neutral NPZ.
# Full model has 10475 vertices and many shapedirs; in our
# attached package the full file has 400 shapedir channels.
# ------------------------------------------------------------
candidates = list(Path("/kaggle/input").rglob("SMPLX_NEUTRAL.npz"))

valid = []
for p in candidates:
    try:
        z = np.load(p, allow_pickle=True)
        vshape = tuple(z["v_template"].shape) if "v_template" in z.files else None
        nshapes = int(z["shapedirs"].shape[-1]) if "shapedirs" in z.files else -1
        z.close()
        print(p, "| MB =", round(p.stat().st_size / 1024**2, 2),
              "| v_template =", vshape, "| shapedirs =", nshapes)
        if vshape == (10475, 3) and nshapes >= 100:
            valid.append((nshapes, p.stat().st_size, p))
    except Exception as e:
        print("Could not inspect:", p, e)

if not valid:
    raise FileNotFoundError(
        "No full SMPLX_NEUTRAL.npz found. Attach the private SMPL-X v1.1 Kaggle Model."
    )

valid.sort(reverse=True)
SMPLX_SOURCE = valid[0][2]
SMPLX_TARGET = REPO / "models" / "smplx" / "SMPLX_NEUTRAL.npz"
SMPLX_TARGET.parent.mkdir(parents=True, exist_ok=True)

if not SMPLX_TARGET.exists() or SMPLX_TARGET.stat().st_size != SMPLX_SOURCE.stat().st_size:
    shutil.copy2(SMPLX_SOURCE, SMPLX_TARGET)

print("\nSMPL-X source:", SMPLX_SOURCE)
print("SMPL-X target:", SMPLX_TARGET)
print("SMPLX_ASSET_READY=PASS")

# ------------------------------------------------------------
# 2) Discover Drishti videos
# ------------------------------------------------------------
video_exts = {".mp4", ".mkv", ".avi", ".mov"}

if PREFERRED_VIDEO_ROOT.exists():
    videos = sorted([
        p for p in PREFERRED_VIDEO_ROOT.rglob("*")
        if p.is_file() and p.suffix.lower() in video_exts
    ])
else:
    videos = sorted([
        p for p in Path("/kaggle/input").rglob("*")
        if p.is_file() and p.suffix.lower() in video_exts
        and "drishti" in str(p).lower()
    ])

assert videos, "No Drishti videos found."

video_rows = []
for p in videos:
    cap = cv2.VideoCapture(str(p))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()
    duration = frames / fps if fps > 0 else float("nan")
    video_rows.append({
        "video": p.name,
        "path": str(p),
        "fps": fps,
        "frames": frames,
        "duration_sec": duration,
        "width": width,
        "height": height,
    })

video_df = pd.DataFrame(video_rows)
display(video_df)

video_manifest = OUTPUT_ROOT / "video_manifest.csv"
video_df.to_csv(video_manifest, index=False)
print("Saved:", video_manifest)


In [ ]:

# ============================================================
# CELL 4 — WRITE THE RESUMABLE MULTI-HMR VIDEO RUNNER
# ============================================================

from pathlib import Path
import ast

runner_path = OUTPUT_ROOT / "run_multihmr_pipeline.py"

RUNNER_CODE = r'''
import os
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import sys
import csv
import json
import math
import time
import argparse
import urllib.request
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
from PIL import Image, ImageOps, ImageDraw

import torch
from scipy.optimize import linear_sum_assignment

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--repo", required=True)
    p.add_argument("--output", required=True)
    p.add_argument("--model-name", default="multiHMR_672_S")
    p.add_argument("--sample-fps", type=float, default=1.0)
    p.add_argument("--det-thresh", type=float, default=0.3)
    p.add_argument("--nms-kernel", type=int, default=3)
    p.add_argument("--fov", type=float, default=60.0)
    p.add_argument("--videos-json", required=True)
    p.add_argument("--worker-id", type=int, default=0)
    p.add_argument("--debug-every", type=int, default=0)
    p.add_argument("--raw-every", type=int, default=0)
    p.add_argument("--smoke-only", type=int, default=0)
    p.add_argument("--smoke-time", type=float, default=140.567)
    return p.parse_args()

args = parse_args()
REPO = Path(args.repo)
OUT = Path(args.output)
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

from model import Model
from utils.constants import MEAN_PARAMS, CACHE_DIR_MULTIHMR
from utils.humans import get_smplx_joint_names

DEVICE = torch.device("cuda")
assert torch.cuda.is_available()
torch.backends.cudnn.benchmark = True

IMG_NORM_MEAN = np.asarray([0.485, 0.456, 0.406], dtype=np.float32).reshape(3,1,1)
IMG_NORM_STD  = np.asarray([0.229, 0.224, 0.225], dtype=np.float32).reshape(3,1,1)

def normalize_rgb(img):
    x = img.astype(np.float32) / 255.0
    x = np.transpose(x, (2, 0, 1))
    x = (x - IMG_NORM_MEAN) / IMG_NORM_STD
    return x.astype(np.float32)

def focal_from_fov(fov, img_size):
    return img_size / (2.0 * np.tan(np.radians(fov) / 2.0))

def camera_matrix(img_size, fov):
    K = torch.eye(3, dtype=torch.float32)
    focal = focal_from_fov(fov, img_size)
    K[0,0] = focal
    K[1,1] = focal
    K[0,2] = img_size // 2
    K[1,2] = img_size // 2
    return K.unsqueeze(0).to(DEVICE)

def tensorize_bgr(frame_bgr, img_size):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    pil = ImageOps.contain(pil, (img_size, img_size))
    pil = ImageOps.pad(pil, size=(img_size, img_size), color=(0, 0, 0))
    arr = np.asarray(pil)
    x = torch.from_numpy(normalize_rgb(arr)).unsqueeze(0).to(DEVICE)
    return x, arr

def download(urls, dst):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    last = None
    for url in urls:
        try:
            print("Downloading:", url, "->", dst, flush=True)
            urllib.request.urlretrieve(url, dst)
            if dst.exists() and dst.stat().st_size > 1024:
                return
        except Exception as e:
            last = e
            print("Download failed:", e, flush=True)
    raise RuntimeError(f"Could not download {dst}: {last}")

def ensure_assets():
    mean_path = Path(MEAN_PARAMS)
    if not mean_path.exists() or mean_path.stat().st_size < 1024:
        download([
            "https://openmmlab-share.oss-cn-hangzhou.aliyuncs.com/mmhuman3d/models/smpl_mean_params.npz?versionId=CAEQHhiBgICN6M3V6xciIDU1MzUzNjZjZGNiOTQ3OWJiZTJmNThiZmY4NmMxMTM4"
        ], mean_path)

    ckpt = Path(CACHE_DIR_MULTIHMR) / f"{args.model_name}.pt"
    if not ckpt.exists() or ckpt.stat().st_size < 1_000_000:
        download([
            f"https://download.europe.naverlabs.com/multihmr/{args.model_name}.pt",
            f"http://download.europe.naverlabs.com/multihmr/{args.model_name}.pt",
        ], ckpt)
    return ckpt

def load_model(ckpt_path):
    print("Loading checkpoint:", ckpt_path, flush=True)
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
    kwargs = dict(vars(ckpt["args"]))
    kwargs["type"] = ckpt["args"].train_return_type
    kwargs["img_size"] = ckpt["args"].img_size[0]
    model = Model(**kwargs).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"], strict=False)
    model.eval()
    return model

@torch.no_grad()
def infer(model, frame_bgr):
    x, padded_rgb = tensorize_bgr(frame_bgr, model.img_size)
    K = camera_matrix(model.img_size, args.fov)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.cuda.amp.autocast(enabled=True):
        humans = model(
            x,
            is_training=False,
            nms_kernel_size=int(args.nms_kernel),
            det_thresh=float(args.det_thresh),
            K=K,
        )
    torch.cuda.synchronize()
    ms = 1000.0 * (time.perf_counter() - t0)
    return humans, padded_rgb, K, ms

JOINT_NAMES = list(get_smplx_joint_names())
JOINT_INDEX = {str(n).lower(): i for i, n in enumerate(JOINT_NAMES)}

def jidx(name):
    return JOINT_INDEX.get(name.lower())

IDX = {
    "pelvis": jidx("pelvis"),
    "neck": jidx("neck"),
    "head": jidx("head"),
    "left_shoulder": jidx("left_shoulder"),
    "right_shoulder": jidx("right_shoulder"),
    "left_wrist": jidx("left_wrist"),
    "right_wrist": jidx("right_wrist"),
}
print("Resolved joints:", IDX, flush=True)

BODY_ROT_NAMES = JOINT_NAMES[:22]
BODY_ROT_INDEX = {str(n).lower(): i for i, n in enumerate(BODY_ROT_NAMES)}

PERSON_COLUMNS = [
    "video","frame_idx","timestamp_sec","track_id","person_idx","score",
    "loc_x_norm","loc_y_norm","pelvis_x","pelvis_y","pelvis_z",
    "shoulder_width_3d","torso_x_ratio","torso_depth_ratio",
    "left_wrist_head_norm","right_wrist_head_norm",
    "left_wrist_neck_norm","right_wrist_neck_norm",
    "left_wrist_pelvis_norm","right_wrist_pelvis_norm",
    "left_wrist_speed_2d","right_wrist_speed_2d","head_speed_2d",
    "global_rot_x","global_rot_y","global_rot_z",
    "neck_rot_x","neck_rot_y","neck_rot_z",
    "head_rot_x","head_rot_y","head_rot_z",
]

FRAME_COLUMNS = [
    "video","frame_idx","timestamp_sec","source_fps","effective_sample_fps",
    "inference_ms","num_people","score_max","score_mean",
    "pelvis_z_min","pelvis_z_mean",
    "wrist_head_norm_min","wrist_neck_norm_min","wrist_pelvis_norm_min",
    "wrist_speed_2d_max","head_speed_2d_max",
    "abs_global_rot_y_max","abs_neck_rot_y_max","abs_head_rot_y_max",
    "torso_x_ratio_max","torso_depth_ratio_max",
]

def arr(x):
    if isinstance(x, torch.Tensor):
        return x.detach().float().cpu().numpy()
    return np.asarray(x)

def safe_norm(v):
    return float(np.linalg.norm(v))

def get_joint(a, name):
    idx = IDX.get(name)
    if idx is None or idx >= len(a):
        return None
    return a[idx]

def rot_feature(rotvec, name):
    idx = BODY_ROT_INDEX.get(name.lower())
    if idx is None or idx >= len(rotvec):
        return [np.nan, np.nan, np.nan]
    return [float(v) for v in rotvec[idx]]

class Tracker:
    def __init__(self, max_dist=0.16, max_missed=3):
        self.max_dist = max_dist
        self.max_missed = max_missed
        self.next_id = 1
        self.tracks = {}

    def assign(self, locs, timestamp):
        ids = [None] * len(locs)
        active_ids = list(self.tracks.keys())

        if active_ids and locs:
            prev = np.asarray([self.tracks[t]["loc"] for t in active_ids], dtype=float)
            cur = np.asarray(locs, dtype=float)
            cost = np.linalg.norm(prev[:,None,:] - cur[None,:,:], axis=2)
            rr, cc = linear_sum_assignment(cost)
            used_cur = set()
            matched_tracks = set()
            for r, c in zip(rr, cc):
                if cost[r,c] <= self.max_dist:
                    tid = active_ids[r]
                    ids[c] = tid
                    used_cur.add(c)
                    matched_tracks.add(tid)

            for tid in active_ids:
                if tid not in matched_tracks:
                    self.tracks[tid]["missed"] += 1

        for i in range(len(locs)):
            if ids[i] is None:
                ids[i] = self.next_id
                self.tracks[self.next_id] = {
                    "loc": np.asarray(locs[i], dtype=float),
                    "timestamp": timestamp,
                    "missed": 0,
                    "left_wrist_2d": None,
                    "right_wrist_2d": None,
                    "head_2d": None,
                }
                self.next_id += 1

        # Update loc/timestamp for assigned tracks.
        for i, tid in enumerate(ids):
            self.tracks[tid]["loc"] = np.asarray(locs[i], dtype=float)
            self.tracks[tid]["missed"] = 0

        # Prune stale.
        stale = [tid for tid,s in self.tracks.items() if s["missed"] > self.max_missed]
        for tid in stale:
            self.tracks.pop(tid, None)

        return ids

def person_features(h, video, frame_idx, timestamp, track_id, person_idx, model_img_size, tracker):
    score = float(arr(h["scores"]).reshape(-1)[0])
    loc = arr(h["loc"]).reshape(-1)[:2] / float(model_img_size)
    j3d = arr(h["j3d"])
    j2d = arr(h["j2d"])
    rotvec = arr(h["rotvec"])
    pelvis_t = arr(h["transl_pelvis"]).reshape(-1)[:3]

    pelvis = get_joint(j3d, "pelvis")
    neck = get_joint(j3d, "neck")
    head = get_joint(j3d, "head")
    ls = get_joint(j3d, "left_shoulder")
    rs = get_joint(j3d, "right_shoulder")
    lw = get_joint(j3d, "left_wrist")
    rw = get_joint(j3d, "right_wrist")

    shoulder_width = safe_norm(ls-rs) if ls is not None and rs is not None else np.nan
    scale = shoulder_width if np.isfinite(shoulder_width) and shoulder_width > 1e-6 else 1.0

    def ndist(a, b):
        if a is None or b is None:
            return np.nan
        return safe_norm(a-b) / scale

    torso_x_ratio = np.nan
    torso_depth_ratio = np.nan
    if neck is not None and pelvis is not None:
        tv = neck - pelvis
        tn = safe_norm(tv) + 1e-8
        torso_x_ratio = abs(float(tv[0])) / tn
        torso_depth_ratio = abs(float(tv[2])) / tn

    # 2D motion features
    state = tracker.tracks[track_id]
    prev_t = state.get("timestamp")
    dt = max(timestamp - prev_t, 1e-6) if prev_t is not None else np.nan

    def j2(name):
        idx = IDX.get(name)
        if idx is None or idx >= len(j2d):
            return None
        return j2d[idx] / float(model_img_size)

    lw2 = j2("left_wrist")
    rw2 = j2("right_wrist")
    hd2 = j2("head")

    def speed(curr, key):
        prev = state.get(key)
        if curr is None or prev is None or not np.isfinite(dt):
            return np.nan
        return safe_norm(curr - prev) / dt

    lw_speed = speed(lw2, "left_wrist_2d")
    rw_speed = speed(rw2, "right_wrist_2d")
    hd_speed = speed(hd2, "head_2d")

    state["left_wrist_2d"] = None if lw2 is None else np.asarray(lw2)
    state["right_wrist_2d"] = None if rw2 is None else np.asarray(rw2)
    state["head_2d"] = None if hd2 is None else np.asarray(hd2)
    state["timestamp"] = timestamp

    grot = [float(v) for v in rotvec[0]] if len(rotvec) else [np.nan]*3
    nrot = rot_feature(rotvec, "neck")
    hrot = rot_feature(rotvec, "head")

    return {
        "video": video,
        "frame_idx": int(frame_idx),
        "timestamp_sec": float(timestamp),
        "track_id": int(track_id),
        "person_idx": int(person_idx),
        "score": score,
        "loc_x_norm": float(loc[0]),
        "loc_y_norm": float(loc[1]),
        "pelvis_x": float(pelvis_t[0]),
        "pelvis_y": float(pelvis_t[1]),
        "pelvis_z": float(pelvis_t[2]),
        "shoulder_width_3d": float(shoulder_width) if np.isfinite(shoulder_width) else np.nan,
        "torso_x_ratio": torso_x_ratio,
        "torso_depth_ratio": torso_depth_ratio,
        "left_wrist_head_norm": ndist(lw, head),
        "right_wrist_head_norm": ndist(rw, head),
        "left_wrist_neck_norm": ndist(lw, neck),
        "right_wrist_neck_norm": ndist(rw, neck),
        "left_wrist_pelvis_norm": ndist(lw, pelvis),
        "right_wrist_pelvis_norm": ndist(rw, pelvis),
        "left_wrist_speed_2d": lw_speed,
        "right_wrist_speed_2d": rw_speed,
        "head_speed_2d": hd_speed,
        "global_rot_x": grot[0], "global_rot_y": grot[1], "global_rot_z": grot[2],
        "neck_rot_x": nrot[0], "neck_rot_y": nrot[1], "neck_rot_z": nrot[2],
        "head_rot_x": hrot[0], "head_rot_y": hrot[1], "head_rot_z": hrot[2],
    }

def nanagg(rows, cols, fn):
    vals = []
    for r in rows:
        for c in cols:
            v = r.get(c, np.nan)
            if v is not None and np.isfinite(v):
                vals.append(float(v))
    if not vals:
        return np.nan
    return float(fn(vals))

def frame_features(rows, video, frame_idx, timestamp, source_fps, eff_fps, inference_ms):
    return {
        "video": video,
        "frame_idx": int(frame_idx),
        "timestamp_sec": float(timestamp),
        "source_fps": float(source_fps),
        "effective_sample_fps": float(eff_fps),
        "inference_ms": float(inference_ms),
        "num_people": len(rows),
        "score_max": nanagg(rows, ["score"], max),
        "score_mean": nanagg(rows, ["score"], np.mean),
        "pelvis_z_min": nanagg(rows, ["pelvis_z"], min),
        "pelvis_z_mean": nanagg(rows, ["pelvis_z"], np.mean),
        "wrist_head_norm_min": nanagg(rows, ["left_wrist_head_norm","right_wrist_head_norm"], min),
        "wrist_neck_norm_min": nanagg(rows, ["left_wrist_neck_norm","right_wrist_neck_norm"], min),
        "wrist_pelvis_norm_min": nanagg(rows, ["left_wrist_pelvis_norm","right_wrist_pelvis_norm"], min),
        "wrist_speed_2d_max": nanagg(rows, ["left_wrist_speed_2d","right_wrist_speed_2d"], max),
        "head_speed_2d_max": nanagg(rows, ["head_speed_2d"], max),
        "abs_global_rot_y_max": nanagg(rows, ["global_rot_y"], lambda x: np.max(np.abs(x))),
        "abs_neck_rot_y_max": nanagg(rows, ["neck_rot_y"], lambda x: np.max(np.abs(x))),
        "abs_head_rot_y_max": nanagg(rows, ["head_rot_y"], lambda x: np.max(np.abs(x))),
        "torso_x_ratio_max": nanagg(rows, ["torso_x_ratio"], max),
        "torso_depth_ratio_max": nanagg(rows, ["torso_depth_ratio"], max),
    }

def append_csv(path, columns, row):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    exists = path.exists() and path.stat().st_size > 0
    with path.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=columns)
        if not exists:
            w.writeheader()
        w.writerow({k: row.get(k, np.nan) for k in columns})

def safe_slug(name):
    keep = []
    for ch in name:
        keep.append(ch if ch.isalnum() or ch in "._-" else "_")
    return "".join(keep)

def draw_debug(padded_rgb, humans, out_path):
    im = Image.fromarray(padded_rgb.copy())
    d = ImageDraw.Draw(im)
    for h in humans:
        pts = arr(h["j2d"])
        for p in pts[:127]:
            x, y = float(p[0]), float(p[1])
            if np.isfinite(x) and np.isfinite(y):
                d.ellipse((x-1.5,y-1.5,x+1.5,y+1.5), fill=(255,255,0))
    im.save(out_path)

def save_raw(humans, out_path):
    data = {}
    for i,h in enumerate(humans):
        for key in ("j3d","j2d","rotvec","shape","expression","transl","transl_pelvis","loc","scores"):
            if key in h:
                data[f"p{i}_{key}"] = arr(h[key])
    np.savez_compressed(out_path, **data)

def process_video(model, path):
    path = Path(path)
    slug = safe_slug(path.name)
    feat_final = OUT / "frame_features" / f"{slug}.csv"
    person_final = OUT / "person_features" / f"{slug}.csv"
    done = OUT / "done" / f"{slug}.done"

    if done.exists() and feat_final.exists():
        print("SKIP complete:", path.name, flush=True)
        return

    feat_partial = OUT / "frame_features" / f"{slug}.partial.csv"
    person_partial = OUT / "person_features" / f"{slug}.partial.csv"
    debug_dir = OUT / "debug" / slug
    raw_dir = OUT / "raw" / slug
    debug_dir.mkdir(parents=True, exist_ok=True)
    raw_dir.mkdir(parents=True, exist_ok=True)
    done.parent.mkdir(parents=True, exist_ok=True)

    processed = set()
    if feat_partial.exists():
        with feat_partial.open() as f:
            for r in csv.DictReader(f):
                try:
                    processed.add(int(r["frame_idx"]))
                except:
                    pass
        print("Resume frames:", len(processed), path.name, flush=True)

    cap = cv2.VideoCapture(str(path))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if fps <= 0:
        raise RuntimeError(f"Invalid FPS for {path}")

    step = max(1, int(round(fps / args.sample_fps)))
    eff_fps = fps / step

    tracker = Tracker()
    times = []
    sample_count = 0
    frame_idx = -1

    torch.cuda.reset_peak_memory_stats()

    while True:
        ok = cap.grab()
        if not ok:
            break
        frame_idx += 1

        if frame_idx % step != 0:
            continue
        if frame_idx in processed:
            continue

        ok, frame = cap.retrieve()
        if not ok or frame is None:
            continue

        timestamp = frame_idx / fps
        humans, padded_rgb, K, ms = infer(model, frame)
        times.append(ms)

        locs = []
        for h in humans:
            loc = arr(h["loc"]).reshape(-1)[:2] / float(model.img_size)
            locs.append(loc)

        tids = tracker.assign(locs, timestamp)

        prows = []
        for i, (h, tid) in enumerate(zip(humans, tids)):
            pr = person_features(
                h, path.name, frame_idx, timestamp, tid, i,
                model.img_size, tracker
            )
            prows.append(pr)
            append_csv(person_partial, PERSON_COLUMNS, pr)

        fr = frame_features(
            prows, path.name, frame_idx, timestamp, fps, eff_fps, ms
        )
        append_csv(feat_partial, FRAME_COLUMNS, fr)

        sample_count += 1

        if args.debug_every > 0 and sample_count % args.debug_every == 0:
            draw_debug(
                padded_rgb, humans,
                debug_dir / f"{frame_idx:08d}_{timestamp:.2f}s.jpg"
            )

        if args.raw_every > 0 and sample_count % args.raw_every == 0:
            save_raw(
                humans,
                raw_dir / f"{frame_idx:08d}_{timestamp:.2f}s.npz"
            )

        if sample_count % 50 == 0:
            med = float(np.median(times[-50:])) if times else np.nan
            print(
                f"[W{args.worker_id}] {path.name} "
                f"{frame_idx}/{total} t={timestamp:.1f}s "
                f"people={len(humans)} med50={med:.1f}ms",
                flush=True
            )

    cap.release()

    # Atomically promote partial files to final.
    feat_final.parent.mkdir(parents=True, exist_ok=True)
    person_final.parent.mkdir(parents=True, exist_ok=True)
    if feat_partial.exists():
        feat_partial.replace(feat_final)
    if person_partial.exists():
        person_partial.replace(person_final)

    runtime = {
        "video": path.name,
        "source_fps": fps,
        "effective_sample_fps": eff_fps,
        "processed_samples_this_run": sample_count,
        "median_inference_ms_this_run": float(np.median(times)) if times else None,
        "p95_inference_ms_this_run": float(np.percentile(times, 95)) if times else None,
        "model_fps_from_median": float(1000.0/np.median(times)) if times and np.median(times) > 0 else None,
        "peak_allocated_mb": float(torch.cuda.max_memory_allocated()/1024**2),
        "peak_reserved_mb": float(torch.cuda.max_memory_reserved()/1024**2),
        "gpu": torch.cuda.get_device_name(0),
    }
    rt_path = OUT / "runtime" / f"{slug}.json"
    rt_path.parent.mkdir(parents=True, exist_ok=True)
    rt_path.write_text(json.dumps(runtime, indent=2))
    done.write_text("PASS\n")
    print("DONE:", path.name, runtime, flush=True)

def smoke(model, path, t_sec):
    path = Path(path)
    cap = cv2.VideoCapture(str(path))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    if fps <= 0:
        raise RuntimeError("Invalid smoke video FPS")
    cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, t_sec) * 1000.0)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError("Could not read smoke frame")

    humans, padded_rgb, K, ms = infer(model, frame)
    smoke_dir = OUT / "smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    draw_debug(padded_rgb, humans, smoke_dir / "smoke_joints.jpg")
    save_raw(humans, smoke_dir / "smoke_raw.npz")

    summary = {
        "video": path.name,
        "requested_time_sec": t_sec,
        "people": len(humans),
        "inference_ms": ms,
        "gpu": torch.cuda.get_device_name(0),
        "human_keys": sorted(list(humans[0].keys())) if humans else [],
    }
    (smoke_dir / "smoke_summary.json").write_text(json.dumps(summary, indent=2))
    print("SMOKE_RESULT:", json.dumps(summary, indent=2), flush=True)

with open(args.videos_json) as f:
    video_paths = json.load(f)

assert video_paths, "videos-json is empty"

ckpt = ensure_assets()
model = load_model(ckpt)
print("MODEL_READY:", args.model_name, "img_size=", model.img_size, flush=True)

if args.smoke_only:
    smoke(model, video_paths[0], args.smoke_time)
    sys.exit(0)

for vp in video_paths:
    process_video(model, vp)

print(f"WORKER_{args.worker_id}_COMPLETE", flush=True)
'''

# The placeholder is replaced when this notebook was generated.
assert "__RUNNER_PLACEHOLDER__" not in RUNNER_CODE

ast.parse(RUNNER_CODE)  # fail now if syntax was damaged
runner_path.write_text(RUNNER_CODE)

print("Runner written:", runner_path)
print("RUNNER_SYNTAX=PASS")


In [ ]:

# ============================================================
# CELL 5 — REAL SMOKE INFERENCE, THEN FULL PARALLEL EXTRACTION
# JUST RUN THIS CELL AND LEAVE IT RUNNING
# ============================================================

import json
import os
import subprocess
from pathlib import Path

video_paths = [str(p) for p in videos]

# Prefer VIDEO_05 / CCTV Mobile Usage for the 140.567 sec smoke frame.
smoke_candidates = [
    p for p in video_paths
    if "CCTV Mobile Usage" in Path(p).name
]
smoke_video = smoke_candidates[0] if smoke_candidates else video_paths[0]

smoke_json = OUTPUT_ROOT / "smoke_video.json"
smoke_json.write_text(json.dumps([smoke_video], indent=2))

base_cmd = [
    str(ENV_PY),
    str(runner_path),
    "--repo", str(REPO),
    "--output", str(OUTPUT_ROOT),
    "--model-name", MODEL_NAME,
    "--sample-fps", str(SAMPLE_FPS),
    "--det-thresh", str(DET_THRESH),
    "--nms-kernel", str(NMS_KERNEL_SIZE),
    "--fov", str(FOV_DEG),
    "--debug-every", str(DEBUG_EVERY_N_SAMPLES),
    "--raw-every", str(RAW_EVERY_N_SAMPLES),
]

# ------------------------------------------------------------
# 1) Smoke first. This also warms checkpoint / torch hub caches.
# ------------------------------------------------------------
print("=" * 100)
print("SMOKE INFERENCE")
print("=" * 100)

smoke_cmd = base_cmd + [
    "--videos-json", str(smoke_json),
    "--worker-id", "0",
    "--smoke-only", "1",
    "--smoke-time", "140.567",
]

env0 = os.environ.copy()
env0["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.run(smoke_cmd, env=env0, check=True)

print("\n✅ SMOKE PASS — proceeding to full extraction.")

# ------------------------------------------------------------
# 2) Detect visible GPUs and split videos.
# ------------------------------------------------------------
gpu_lines = subprocess.run(
    ["nvidia-smi", "-L"],
    capture_output=True,
    text=True,
    check=True
).stdout.strip().splitlines()

num_gpus = max(1, min(len(gpu_lines), MAX_GPU_WORKERS, len(video_paths)))
print("GPU workers:", num_gpus)

parts = [[] for _ in range(num_gpus)]
for i, vp in enumerate(video_paths):
    parts[i % num_gpus].append(vp)

workers = []

for gpu_id, part in enumerate(parts):
    part_json = OUTPUT_ROOT / f"worker_{gpu_id}_videos.json"
    part_json.write_text(json.dumps(part, indent=2))

    cmd = base_cmd + [
        "--videos-json", str(part_json),
        "--worker-id", str(gpu_id),
    ]

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

    print(f"\nLaunching worker {gpu_id} on GPU {gpu_id}:")
    for p in part:
        print("  -", Path(p).name)

    workers.append((gpu_id, subprocess.Popen(cmd, env=env)))

failures = []
for gpu_id, proc in workers:
    code = proc.wait()
    print(f"Worker {gpu_id} exit code:", code)
    if code != 0:
        failures.append((gpu_id, code))

if failures:
    raise RuntimeError(f"One or more inference workers failed: {failures}")

print("\n" + "=" * 100)
print("✅ FULL MULTI-HMR EXTRACTION COMPLETE")
print("=" * 100)


In [ ]:

# ============================================================
# CELL 6 — MERGE EXTRACTED FEATURES + LOAD/CREATE ANNOTATIONS
# ============================================================

import json
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Merge per-video frame/person feature files.
# ------------------------------------------------------------
frame_files = sorted((OUTPUT_ROOT / "frame_features").glob("*.csv"))
person_files = sorted((OUTPUT_ROOT / "person_features").glob("*.csv"))

assert frame_files, "No completed frame feature CSVs found."

frame_df = pd.concat([pd.read_csv(p) for p in frame_files], ignore_index=True)
frame_df = frame_df.sort_values(["video", "timestamp_sec"]).reset_index(drop=True)

if person_files:
    person_df = pd.concat([pd.read_csv(p) for p in person_files], ignore_index=True)
    person_df = person_df.sort_values(["video", "timestamp_sec", "track_id"]).reset_index(drop=True)
else:
    person_df = pd.DataFrame()

frame_merged = OUTPUT_ROOT / "frame_features_merged.csv"
person_merged = OUTPUT_ROOT / "person_features_merged.csv"
frame_df.to_csv(frame_merged, index=False)
person_df.to_csv(person_merged, index=False)

print("Frame rows:", len(frame_df))
print("Person rows:", len(person_df))
print("Videos extracted:", frame_df["video"].nunique())

# ------------------------------------------------------------
# 2) Resolve annotations.
# Priority: MANUAL_EVENTS -> uploaded CSV -> template only.
# Expected columns: video,start_sec,end_sec,label
# ------------------------------------------------------------
ANNOTATIONS_PATH = None

if MANUAL_EVENTS:
    ann = pd.DataFrame(MANUAL_EVENTS)
    required = {"video","start_sec","end_sec","label"}
    missing = required - set(ann.columns)
    assert not missing, f"MANUAL_EVENTS missing columns: {missing}"
    ANNOTATIONS_PATH = OUTPUT_ROOT / "annotations_manual.csv"
    ann.to_csv(ANNOTATIONS_PATH, index=False)
else:
    preferred_names = {
        "drishti_annotations.csv",
        "annotations.csv",
        "events.csv",
        "temporal_annotations.csv",
    }
    candidates = [
        p for p in Path("/kaggle/input").rglob("*.csv")
        if p.name.lower() in preferred_names
    ]
    if candidates:
        ANNOTATIONS_PATH = candidates[0]

if ANNOTATIONS_PATH is None:
    template = OUTPUT_ROOT / "annotations_TEMPLATE.csv"
    pd.DataFrame(columns=[
        "video","start_sec","end_sec","label","notes"
    ]).to_csv(template, index=False)

    print("\n⚠️ NO REAL TEMPORAL ANNOTATIONS FOUND.")
    print("Multi-HMR extraction is complete, but supervised training will not fabricate labels.")
    print("Fill/upload:")
    print(template)
else:
    print("Annotations:", ANNOTATIONS_PATH)
    ann = pd.read_csv(ANNOTATIONS_PATH)
    display(ann.head(20))


In [ ]:

# ============================================================
# CELL 7 — OPTIONAL OBJECT-EVIDENCE FUSION + TEMPORAL FEATURES
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

features = frame_df.copy()

# ------------------------------------------------------------
# 1) Optional hook for teammate object evidence.
# If present, expected minimum columns:
# video,timestamp_sec
# Other numeric fields (phone_conf, mouse_conf, monitor_conf, etc.)
# are merged automatically and can become classifier features.
# ------------------------------------------------------------
object_candidates = list(Path("/kaggle/working").rglob("object_evidence.csv"))
object_candidates += list(Path("/kaggle/input").rglob("object_evidence.csv"))

if object_candidates:
    obj_path = object_candidates[0]
    obj = pd.read_csv(obj_path)
    if {"video","timestamp_sec"}.issubset(obj.columns):
        print("Merging external object evidence:", obj_path)
        merged_parts = []
        for video, fpart in features.groupby("video", sort=False):
            opart = obj[obj["video"] == video].sort_values("timestamp_sec")
            fpart = fpart.sort_values("timestamp_sec")
            if len(opart):
                m = pd.merge_asof(
                    fpart,
                    opart,
                    on="timestamp_sec",
                    by="video",
                    direction="nearest",
                    tolerance=max(0.51 / SAMPLE_FPS, 0.25),
                    suffixes=("","_obj"),
                )
            else:
                m = fpart
            merged_parts.append(m)
        features = pd.concat(merged_parts, ignore_index=True)
    else:
        print("object_evidence.csv ignored: missing video,timestamp_sec.")
else:
    print("No object_evidence.csv found — pose-only baseline will train.")

# ------------------------------------------------------------
# 2) Rolling temporal features, within each video only.
# ------------------------------------------------------------
meta_exclude = {
    "frame_idx","timestamp_sec","source_fps","effective_sample_fps","inference_ms"
}

base_numeric = [
    c for c in features.select_dtypes(include=[np.number]).columns
    if c not in meta_exclude
]

window_n = max(2, int(round(TEMPORAL_WINDOW_SEC * SAMPLE_FPS)))
print("Temporal rolling window samples:", window_n)

features = features.sort_values(["video","timestamp_sec"]).reset_index(drop=True)

for c in base_numeric:
    g = features.groupby("video", sort=False)[c]
    features[f"{c}__roll_mean"] = g.transform(
        lambda s: s.rolling(window_n, min_periods=1).mean()
    )
    features[f"{c}__roll_std"] = g.transform(
        lambda s: s.rolling(window_n, min_periods=2).std()
    )
    features[f"{c}__roll_max"] = g.transform(
        lambda s: s.rolling(window_n, min_periods=1).max()
    )
    features[f"{c}__delta"] = g.diff()

temporal_path = OUTPUT_ROOT / "temporal_features.csv"
features.to_csv(temporal_path, index=False)

print("Temporal feature shape:", features.shape)
print("Saved:", temporal_path)


In [ ]:

# ============================================================
# CELL 8 — LABEL FROM REAL INTERVALS + GROUPED TRAIN/VAL/TEST
# ============================================================

import json
import math
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
)

if ANNOTATIONS_PATH is None:
    print("TRAINING_SKIPPED_NO_GROUND_TRUTH=TRUE")
else:
    ann = pd.read_csv(ANNOTATIONS_PATH).copy()
    required = {"video","start_sec","end_sec","label"}
    missing = required - set(ann.columns)
    assert not missing, f"Annotation CSV missing: {missing}"

    ann["video"] = ann["video"].astype(str)
    ann["label"] = ann["label"].astype(str)
    ann["start_sec"] = pd.to_numeric(ann["start_sec"])
    ann["end_sec"] = pd.to_numeric(ann["end_sec"])
    assert (ann["end_sec"] >= ann["start_sec"]).all()

    annotated_videos = set(ann["video"].unique())
    train_df = features[features["video"].isin(annotated_videos)].copy()

    if not len(train_df):
        raise RuntimeError(
            "Annotation video names do not match extracted video basenames."
        )

    # Label frames from temporal intervals.
    train_df["event_label"] = None
    train_df["is_suspicious"] = np.nan

    for video, idxs in train_df.groupby("video").groups.items():
        v_ann = ann[ann["video"] == video]
        times = train_df.loc[idxs, "timestamp_sec"].to_numpy()

        # Only do this if annotation coverage is intended to be complete.
        if ASSUME_OUTSIDE_ANNOTATIONS_NORMAL:
            train_df.loc[idxs, "event_label"] = "normal"
            train_df.loc[idxs, "is_suspicious"] = 0

        for _, r in v_ann.iterrows():
            mask = (times >= r["start_sec"]) & (times <= r["end_sec"])
            rows = np.asarray(list(idxs))[mask]
            label = str(r["label"])
            train_df.loc[rows, "event_label"] = label
            train_df.loc[rows, "is_suspicious"] = (
                0 if label.strip().lower() in NORMAL_LABELS else 1
            )

    # If outside intervals are not assumed normal, unlabeled rows are excluded.
    train_df = train_df[train_df["is_suspicious"].notna()].copy()
    train_df["is_suspicious"] = train_df["is_suspicious"].astype(int)

    print("Labeled rows:", len(train_df))
    print("Labeled videos:", train_df["video"].nunique())
    print(train_df["is_suspicious"].value_counts(dropna=False))
    print(train_df["event_label"].value_counts(dropna=False).head(20))

    if train_df["is_suspicious"].nunique() < 2:
        raise RuntimeError("Need both suspicious and normal labels to train.")

    # --------------------------------------------------------
    # Feature selection: no timestamp/runtime leakage.
    # --------------------------------------------------------
    exclude = {
        "video","frame_idx","timestamp_sec","source_fps","effective_sample_fps",
        "inference_ms","event_label","is_suspicious"
    }
    feature_cols = [
        c for c in train_df.select_dtypes(include=[np.number]).columns
        if c not in exclude
    ]

    X = train_df[feature_cols]
    y = train_df["is_suspicious"].to_numpy()
    groups = train_df["video"].to_numpy()

    def find_group_split(df, test_size, seed0=42, tries=200):
        for seed in range(seed0, seed0 + tries):
            splitter = GroupShuffleSplit(
                n_splits=1, test_size=test_size, random_state=seed
            )
            tr_idx, te_idx = next(
                splitter.split(df, df["is_suspicious"], groups=df["video"])
            )
            tr = df.iloc[tr_idx]
            te = df.iloc[te_idx]
            if tr["is_suspicious"].nunique() == 2 and te["is_suspicious"].nunique() == 2:
                return tr.index.to_numpy(), te.index.to_numpy(), seed
        return None

    if train_df["video"].nunique() < 3:
        raise RuntimeError(
            "Need at least 3 labeled videos for a grouped train/test evaluation."
        )

    outer = find_group_split(train_df, test_size=0.25)
    if outer is None:
        raise RuntimeError(
            "Could not create a video-grouped test split containing both classes. "
            "Add/adjust temporal annotations rather than falling back to random-frame leakage."
        )

    trainval_idx, test_idx, outer_seed = outer
    trainval = train_df.loc[trainval_idx].copy()
    test = train_df.loc[test_idx].copy()

    inner = None
    if trainval["video"].nunique() >= 3:
        inner = find_group_split(trainval, test_size=0.25, seed0=100)

    if inner is None:
        # Small-data mode: hold threshold at 0.5 and compare models via train F1 is not honest.
        # Use logistic baseline only; test remains untouched.
        tr = trainval
        val = None
        print("⚠️ No valid grouped validation split; using fixed 0.5 threshold.")
    else:
        tr_idx, val_idx, inner_seed = inner
        tr = trainval.loc[tr_idx]
        val = trainval.loc[val_idx]

    print("\nTRAIN videos:", sorted(tr["video"].unique()))
    print("VAL videos:", [] if val is None else sorted(val["video"].unique()))
    print("TEST videos:", sorted(test["video"].unique()))

    def binary_metrics(y_true, prob, threshold):
        pred = (prob >= threshold).astype(int)
        p,r,f1,_ = precision_recall_fscore_support(
            y_true, pred, average="binary", zero_division=0
        )
        tn,fp,fn,tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
        normal_minutes = max((y_true == 0).sum() / SAMPLE_FPS / 60.0, 1e-9)
        fp_per_min = fp / normal_minutes
        out = {
            "precision": float(p),
            "recall": float(r),
            "f1": float(f1),
            "fp_per_min": float(fp_per_min),
            "tp": int(tp),"fp": int(fp),"tn": int(tn),"fn": int(fn),
        }
        if len(np.unique(y_true)) == 2:
            out["average_precision"] = float(average_precision_score(y_true, prob))
            out["roc_auc"] = float(roc_auc_score(y_true, prob))
        return out

    candidates = {
        "logistic": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=2500,
                class_weight="balanced",
                C=1.0,
                random_state=42,
            )),
        ]),
        "random_forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=350,
                max_depth=12,
                min_samples_leaf=3,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=42,
            )),
        ]),
    }

    selected_name = "logistic"
    selected_threshold = 0.5

    if val is not None:
        best_key = None
        best_bundle = None

        for name, clf in candidates.items():
            clf.fit(tr[feature_cols], tr["is_suspicious"])
            prob = clf.predict_proba(val[feature_cols])[:,1]

            threshold_results = []
            for thr in np.linspace(0.10, 0.90, 81):
                m = binary_metrics(val["is_suspicious"].to_numpy(), prob, float(thr))
                threshold_results.append((float(thr), m))

            feasible = [
                (thr,m) for thr,m in threshold_results
                if m["fp_per_min"] <= MAX_FP_PER_MIN_TARGET
            ]
            pool = feasible if feasible else threshold_results
            thr, metrics = max(
                pool,
                key=lambda tm: (tm[1]["f1"], tm[1]["recall"], -tm[1]["fp_per_min"])
            )

            key = (metrics["f1"], metrics["recall"], -metrics["fp_per_min"])
            print(name, "VAL", "thr=", round(thr,3), metrics)

            if best_key is None or key > best_key:
                best_key = key
                best_bundle = (name, thr, metrics)

        selected_name, selected_threshold, selected_val_metrics = best_bundle
        print("\nSELECTED:", selected_name, "threshold=", selected_threshold)

    # Retrain chosen model on train+validation only; test still untouched.
    final_model = candidates[selected_name]
    final_model.fit(trainval[feature_cols], trainval["is_suspicious"])

    test_prob = final_model.predict_proba(test[feature_cols])[:,1]
    test_metrics = binary_metrics(
        test["is_suspicious"].to_numpy(),
        test_prob,
        selected_threshold
    )

    print("\n" + "="*90)
    print("FRAME-LEVEL TEST METRICS")
    print("="*90)
    print(json.dumps(test_metrics, indent=2))

    # --------------------------------------------------------
    # Persistence -> predicted event windows
    # --------------------------------------------------------
    test_pred = test[["video","frame_idx","timestamp_sec","is_suspicious","event_label"]].copy()
    test_pred["prob_suspicious"] = test_prob
    test_pred["pred_frame"] = (test_prob >= selected_threshold).astype(int)

    def predicted_events_for_video(dfv):
        dfv = dfv.sort_values("timestamp_sec")
        times = dfv["timestamp_sec"].to_numpy()
        pred = dfv["pred_frame"].to_numpy()
        events = []
        start = None
        last = None
        max_gap = 1.6 / SAMPLE_FPS

        for t, yhat in zip(times, pred):
            if yhat:
                if start is None:
                    start = t
                    last = t
                elif t - last <= max_gap:
                    last = t
                else:
                    if last - start + 1.0/SAMPLE_FPS >= PERSISTENCE_SEC:
                        events.append((start,last))
                    start = t
                    last = t
            elif start is not None:
                if last - start + 1.0/SAMPLE_FPS >= PERSISTENCE_SEC:
                    events.append((start,last))
                start = last = None

        if start is not None and last - start + 1.0/SAMPLE_FPS >= PERSISTENCE_SEC:
            events.append((start,last))
        return events

    predicted_event_rows = []
    for video, dfv in test_pred.groupby("video"):
        for s,e in predicted_events_for_video(dfv):
            predicted_event_rows.append({
                "video": video, "start_sec": float(s), "end_sec": float(e)
            })

    pred_events = pd.DataFrame(predicted_event_rows)

    gt_suspicious = ann[
        (~ann["label"].str.strip().str.lower().isin(NORMAL_LABELS))
        & (ann["video"].isin(test["video"].unique()))
    ][["video","start_sec","end_sec","label"]].copy()

    def overlaps(a0,a1,b0,b1):
        return min(a1,b1) >= max(a0,b0)

    matched_gt = set()
    matched_pred = set()

    for pi, pr in pred_events.iterrows():
        for gi, gr in gt_suspicious.iterrows():
            if pr["video"] != gr["video"]:
                continue
            if overlaps(pr["start_sec"],pr["end_sec"],gr["start_sec"],gr["end_sec"]):
                matched_pred.add(pi)
                matched_gt.add(gi)
                break

    event_tp = len(matched_gt)
    event_fn = len(gt_suspicious) - event_tp
    event_fp = len(pred_events) - len(matched_pred)

    event_precision = event_tp / max(event_tp + event_fp, 1)
    event_recall = event_tp / max(event_tp + event_fn, 1)
    event_f1 = (
        2 * event_precision * event_recall / max(event_precision + event_recall, 1e-12)
    )

    event_metrics = {
        "event_precision": float(event_precision),
        "event_recall": float(event_recall),
        "event_f1": float(event_f1),
        "event_tp": int(event_tp),
        "event_fp": int(event_fp),
        "event_fn": int(event_fn),
        "persistence_sec": float(PERSISTENCE_SEC),
    }

    print("\n" + "="*90)
    print("EVENT-LEVEL TEST METRICS")
    print("="*90)
    print(json.dumps(event_metrics, indent=2))

    # --------------------------------------------------------
    # Persist model + exact inference contract
    # --------------------------------------------------------
    model_dir = OUTPUT_ROOT / "trained"
    model_dir.mkdir(parents=True, exist_ok=True)

    joblib.dump(final_model, model_dir / "behavior_classifier.joblib")
    (model_dir / "feature_columns.json").write_text(json.dumps(feature_cols, indent=2))
    (model_dir / "decision.json").write_text(json.dumps({
        "model_name": selected_name,
        "threshold": float(selected_threshold),
        "temporal_window_sec": float(TEMPORAL_WINDOW_SEC),
        "persistence_sec": float(PERSISTENCE_SEC),
        "sample_fps": float(SAMPLE_FPS),
        "normal_labels": sorted(NORMAL_LABELS),
    }, indent=2))

    metrics_bundle = {
        "frame_test": test_metrics,
        "event_test": event_metrics,
        "train_videos": sorted(tr["video"].unique().tolist()),
        "val_videos": [] if val is None else sorted(val["video"].unique().tolist()),
        "test_videos": sorted(test["video"].unique().tolist()),
        "selected_model": selected_name,
        "threshold": float(selected_threshold),
    }
    (model_dir / "metrics.json").write_text(json.dumps(metrics_bundle, indent=2))

    test_pred.to_csv(model_dir / "test_frame_predictions.csv", index=False)
    pred_events.to_csv(model_dir / "predicted_events.csv", index=False)
    gt_suspicious.to_csv(model_dir / "test_ground_truth_events.csv", index=False)

    print("\n✅ TRAINING_AND_TESTING_COMPLETE=PASS")
    print("Saved model:", model_dir / "behavior_classifier.joblib")


In [ ]:

# ============================================================
# CELL 9 — FINAL D-DAY HANDOFF REPORT
# ============================================================

import json
import pandas as pd
from pathlib import Path

print("=" * 100)
print("DRISHTI MULTI-HMR D-DAY HANDOFF")
print("=" * 100)

print("\nProvider:")
print("  Repo commit:", REPO_COMMIT)
print("  Model:", MODEL_NAME)
print("  Sampling FPS:", SAMPLE_FPS)
print("  SMPL-X:", SMPLX_TARGET)

smoke_summary = OUTPUT_ROOT / "smoke" / "smoke_summary.json"
if smoke_summary.exists():
    print("\nSmoke:")
    print(smoke_summary.read_text())

runtime_files = sorted((OUTPUT_ROOT / "runtime").glob("*.json"))
if runtime_files:
    runtime = pd.DataFrame([json.loads(p.read_text()) for p in runtime_files])
    print("\nRuntime benchmark:")
    display(runtime)

metrics_path = OUTPUT_ROOT / "trained" / "metrics.json"
if metrics_path.exists():
    print("\nClassifier test metrics:")
    print(metrics_path.read_text())
else:
    print("\nClassifier metrics not present.")
    print("If Multi-HMR extraction completed but training was skipped, add real temporal annotations and rerun Cells 6-9.")

print("\nArtifacts:")
for p in [
    OUTPUT_ROOT / "video_manifest.csv",
    OUTPUT_ROOT / "frame_features_merged.csv",
    OUTPUT_ROOT / "person_features_merged.csv",
    OUTPUT_ROOT / "temporal_features.csv",
    OUTPUT_ROOT / "smoke",
    OUTPUT_ROOT / "runtime",
    OUTPUT_ROOT / "trained",
]:
    print(" ", p, "->", "EXISTS" if p.exists() else "missing")

print("\n✅ NOTEBOOK PIPELINE FINISHED")
